# Variantes de denoising con cinco semillas, DS2OS

Compara con el control las configuraciones que lo superan con la semilla 42, además del masking mediante Dropout al 10 %, que es la variante aplicada al tráfico real. Se repite con las semillas 42, 1, 2, 3 y 4. La partición de los datos es fija, de modo que solo varía el azar del entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
DATA_PATH = Path.home() / ".cache/kagglehub/datasets/libamariyam/ds2os-dataset/versions/1/DS2OS.csv"

df = pd.read_csv(DATA_PATH).drop(columns=['timestamp'])
df['accessedNodeType'] = df['accessedNodeType'].fillna('missing')
df['value'] = df['value'].replace({'true': 1, 'false': 0, 'twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)
print(df.shape)

(357952, 12)


In [3]:
from sklearn.model_selection import train_test_split

CAT_COLS = ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
            'destinationServiceAddress', 'destinationServiceType', 'destinationLocation',
            'accessedNodeAddress', 'accessedNodeType', 'operation']

y = (df['normality'] != 'normal').astype(int).values
X = pd.get_dummies(df.drop(columns=['normality']), columns=CAT_COLS)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = MinMaxScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (286361, 505), X_train_normal: (278347, 505), X_test: (71591, 505)


In [4]:
class PureMasking(layers.Layer):
    """Anula una fracción mask_rate de las columnas sin reescalar el resto, a diferencia de Dropout."""
    def __init__(self, mask_rate, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate

    def call(self, inputs, training=None):
        if training:
            keep_mask = tf.cast(tf.random.uniform(tf.shape(inputs)) >= self.mask_rate, inputs.dtype)
            return inputs * keep_mask
        return inputs


CORRUPTION = {
    'gaussiano': layers.GaussianNoise,
    'dropout': layers.Dropout,
    'puro': PureMasking,
}


def build_dae(input_dim, hidden_units, kind=None, level=0.0):
    """Autoencoder denso con una capa de corrupción opcional a la entrada (kind=None es el control)."""
    inputs = keras.Input(shape=(input_dim,))
    x = CORRUPTION[kind](level)(inputs) if kind else inputs
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)

In [5]:
HIDDEN_UNITS = [128, 64, 16]
CONFIGS = [('control', 0.0)] + [('gaussiano', 0.1), ('gaussiano', 0.2), ('dropout', 0.1)]
SEEDS = [42, 1, 2, 3, 4]

results = {}
for seed in SEEDS:
    for kind, level in CONFIGS:
        tf.keras.utils.set_random_seed(seed)
        model = build_dae(X_train.shape[1], HIDDEN_UNITS, None if kind == 'control' else kind, level)
        history = model.fit(
            X_train_normal, X_train_normal,
            epochs=100,
            batch_size=256,
            validation_split=0.1,
            callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
            verbose=0,
        )
        auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
        results[(f"{kind} {level}", seed)] = auc
        print(f"seed={seed} {kind}={level}: epocas={len(history.history['loss'])}, ROC-AUC={auc:.4f}", flush=True)

seed=42 control=0.0: epocas=99, ROC-AUC=0.9211


seed=42 gaussiano=0.1: epocas=100, ROC-AUC=0.9277


seed=42 gaussiano=0.2: epocas=100, ROC-AUC=0.9341


seed=42 dropout=0.1: epocas=100, ROC-AUC=0.9269


seed=1 control=0.0: epocas=28, ROC-AUC=0.8431


...

seed=3 dropout=0.1: epocas=100, ROC-AUC=0.8661


seed=4 control=0.0: epocas=68, ROC-AUC=0.9370


seed=4 gaussiano=0.1: epocas=100, ROC-AUC=0.9347


seed=4 gaussiano=0.2: epocas=100, ROC-AUC=0.7834


seed=4 dropout=0.1: epocas=100, ROC-AUC=0.8952


In [6]:
from scipy.stats import ttest_rel

auc = pd.Series(results).unstack()  # filas: configuración, columnas: semilla
print("AUC por configuracion y semilla:")
print(auc.round(4).to_string())

control = auc.loc['control 0.0']
new = [s for s in SEEDS if s != 42]
print("\nResumen frente al control de la misma semilla:")
for name, row in auc.iterrows():
    line = f"  {name:14s} media={row.mean():.4f} std={row.std(ddof=1):.4f} media 4 nuevas={row[new].mean():.4f}"
    if name != 'control 0.0':
        d = row - control
        line += (f" | dif={d.mean():+.4f} mejora en {int((d > 0).sum())}/5 ({int((d[new] > 0).sum())}/4 nuevas)"
                 f" p={ttest_rel(row, control).pvalue:.3f}")
    print(line)

AUC por configuracion y semilla:
                   1       2       3       4       42
control 0.0    0.8431  0.9168  0.8642  0.9370  0.9211
dropout 0.1    0.9378  0.8880  0.8661  0.8952  0.9269
gaussiano 0.1  0.9109  0.9378  0.9634  0.9347  0.9277
gaussiano 0.2  0.8923  0.7140  0.9601  0.7834  0.9341

Resumen frente al control de la misma semilla:
  control 0.0    media=0.8964 std=0.0405 media 4 nuevas=0.8903
  dropout 0.1    media=0.9028 std=0.0293 media 4 nuevas=0.8968 | dif=+0.0064 mejora en 3/5 (2/4 nuevas) p=0.802
  gaussiano 0.1  media=0.9349 std=0.0190 media 4 nuevas=0.9367 | dif=+0.0384 mejora en 4/5 (3/4 nuevas) p=0.119
  gaussiano 0.2  media=0.8568 std=0.1045 media 4 nuevas=0.8375 | dif=-0.0396 mejora en 3/5 (2/4 nuevas) p=0.536
